In [1]:
import requests,os

url = "https://api.github.com/repos/JayanGupta/Data_Science_Notes/contents/Sprint%201/data/daily_logs"
os.makedirs("daily_logs",exist_ok=True)

for f in requests.get(url).json():
  open("daily_logs/"+f["name"],"wb").write(requests.get(f["download_url"]).content)

In [2]:
import os

LOG_DIR = 'daily_logs'

def is_valid_logfile(filename):
  if filename.startswith(".") or filename.startswith("~"):
    return False

  return filename.lower().endswith(('.txt',".json"))


In [3]:
all = os.listdir(LOG_DIR)
data_files = [os.path.join(LOG_DIR,f) for f in all if is_valid_logfile(f)]
data_files.sort()

In [4]:
print("Total number of files:",len(all))

Total number of files: 74


In [5]:
print("Total number of files:",len(data_files))

Total number of files: 72


###Parsing JSON Log File

In [6]:
import json

def parse_json_file(filepath):
  try:
    with open(filepath ,"r" , encoding = "utf-8") as f:
      data = json.load(f)
    record = {
    "source" :filepath,
    "ip" : data["ip_address"],
    "status" : data["status_code"]
    }
    return record , None

  except(json.JSONDecodeError , KeyError) as e:
    return None , f"{type(e).__name__} : {e}"

In [7]:
sample = [f for f in data_files if f.endswith(".json")][0]
record , error = parse_json_file(sample)
print(f"File:   {os.path.basename(sample)}")
print(f"Record: {record}")
print(f"Error:  {error}")

File:   event_000.json
Record: {'source': 'daily_logs/event_000.json', 'ip': '203.0.113.55', 'status': 200}
Error:  None


###Parsing TEXT Log File

In [8]:
def parse_text_file(filepath):
  try:
    with open(filepath,"r",encoding="utf-8") as f:
      line=f.readline().strip()
      ip,status=line.split(",")
    record={
        "source":filepath,
        "ip": ip.strip(),
        "status" :status.strip()
    }
    return record , None

  except ValueError as e:
    return None ,  f"{type(e).__name__} : {e}"


In [9]:
sample = [f for f in data_files if f.endswith(".txt")][0]
record , error = parse_text_file(sample)
print(f"File:   {os.path.basename(sample)}")
print(f"Record: {record}")
print(f"Error:  {error}")

File:   access_040.txt
Record: {'source': 'daily_logs/access_040.txt', 'ip': '192.168.1.14', 'status': '503'}
Error:  None


In [10]:
class DataParser:
  def __init__(self,filepath):
    self.filepath = filepath
    self.is_valid = True
    self.record = None
    self.error = None

  def parse(self):
    if self.filepath.lower().endswith(".json"):
        self.record , self.error = parse_json_file(self.filepath)

    else:
        self.record , self.error  = parse_text_file(self.filepath)

    if self.error is not None:
        self.is_valid = False

    return self.record


In [11]:
bad = 0

for path in data_files:
  parser = DataParser(path)
  parser.parse()
  if not parser.is_valid:
    bad +=1

print("Bad Files :" , bad)


Bad Files : 7


In [12]:
from collections import Counter

clean_records = []
quarantine_list = []
error_counts = Counter()

for path in data_files:
    parser = DataParser(path)
    parser.parse()

    if parser.is_valid and parser.record is not None:
        clean_records.append(parser.record)

        if str(parser.record["status"]).startswith("5"):
            error_counts[parser.record["ip"]] += 1

    else:
        quarantine_list.append(path)

In [13]:
print(f"Clean records collected:      {len(clean_records)}")
print(f"Corrupt files for quarantine: {len(quarantine_list)}\n")
print("Top 5 Offending IPs (5xx Server Errors):")
for ip, count in error_counts.most_common(5):
    print(f"  IP: {ip:16s} | Errors: {count}")

Clean records collected:      65
Corrupt files for quarantine: 7

Top 5 Offending IPs (5xx Server Errors):
  IP: 45.33.32.156     | Errors: 3
  IP: 198.51.100.23    | Errors: 3
  IP: 10.0.0.8         | Errors: 3
  IP: 192.168.1.14     | Errors: 2
  IP: 203.0.113.55     | Errors: 2


In [14]:
import shutil
# 1. Saving consolidated clean records

OUTPUT_JSON = "analytics_ready.json"
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(clean_records, f, indent=2)
print(f"[OUTPUT] Saved {len(clean_records)} clean records to '{OUTPUT_JSON}'")


[OUTPUT] Saved 65 clean records to 'analytics_ready.json'


In [15]:
# 2. Isolating corrupted files into quarantine directory
QUARANTINE_DIR = "data/quarantine"
os.makedirs(QUARANTINE_DIR, exist_ok=True)

for bad_file in quarantine_list:
    dest = os.path.join(QUARANTINE_DIR, os.path.basename(bad_file))
    shutil.copy(bad_file, dest)

print(f"[QUARANTINE] Isolated {len(quarantine_list)} corrupted files in '{QUARANTINE_DIR}/'")

[QUARANTINE] Isolated 7 corrupted files in 'data/quarantine/'
